# OSM-Netz vorbereiten (gemeinsam für alle Kampagnen)

Lädt den Deutschland-Auszug von Geofabrik, filtert die relevanten Wege mit
`osmium tags-filter` und schreibt GeoParquet nach `utils/processed_osm_files/`.

**Dieses Notebook ersetzt die beiden früheren `0_prepare_network_pbf_get_cycleways.ipynb`**
in `cycleway_complete_campaign/` und `cycleway_complete_marking_campaign/`. Die waren bis auf
zwei Kleinigkeiten gleich und schon auseinandergelaufen: die Marking-Fassung stand auf
OSM-Stand `260701` und kannte `sidewalk:bicycle` nicht. Dazu lagen zwei je ~700 MB große
Kopien nahezu desselben Parquets herum.

## Wer das Ergebnis liest

| Notebook | Kampagne |
| --- | --- |
| `cycleway_complete_campaign/maproulette_tasks.ipynb` | MapRoulette, Radwege |
| `cycleway_complete_marking_campaign/maproulette_tasks.ipynb` | MapRoulette, Markierungen |

Beide lesen `../utils/processed_osm_files/processed_cycleways_germany_{set_date}.parquet`.
Der `set_date` steht dort jeweils oben und muss zu dem hier passen.

> Die wöchentlichen Server-Läufe (`x_`/`xb_` und `2_create_pmtiles`) brauchen dieses
> Notebook **nicht** — sie arbeiten ausschließlich auf `output/*.parquet`.

## Voraussetzungen

- die Python-Umgebung aus `use_cases/pyproject.toml` (`uv sync` in `use_cases/`)
- das [osmium-Kommandozeilenwerkzeug](https://osmcode.org/osmium-tool/), z. B. `sudo apt install osmium-tool`

Eine eigene GDAL-/Conda-Umgebung braucht es nicht: das pyogrio-Wheel bringt GDAL samt
OSM-Treiber mit, und das Parquet schreibt GeoPandas/pyarrow.

In [ ]:
import shutil
import subprocess
from pathlib import Path

import pyarrow.parquet as pq
import pyogrio
import requests

# Muss zum set_date in den 1_/1b_-Notebooks der Kampagnen passen.
# Verfügbare Stände: https://download.geofabrik.de/europe/germany.html
# Achtung: Geofabrik hält Tagesstände nur rund 90 Tage vor.
set_date = "260915"

folder_download = Path("utils/osm_geofabrik_pbf")
folder_processed = Path("utils/processed_osm_files")
source_pbf = folder_download / f"germany-{set_date}.osm.pbf"

overwrite = False  # vorhandene Ergebnisse neu erzeugen
keep_filtered_pbf = False  # Zwischendatei (~470 MB), nur zur Fehlersuche nützlich

folder_download.mkdir(parents=True, exist_ok=True)
folder_processed.mkdir(parents=True, exist_ok=True)

In [ ]:
osmium_binary = shutil.which("osmium")
if not osmium_binary:
    raise FileNotFoundError(
        "`osmium` was not found in PATH. Install osmium-tool, e.g. `sudo apt install osmium-tool`."
    )
if "OSM" not in pyogrio.list_drivers():
    raise RuntimeError("This pyogrio/GDAL build does not include the OSM driver.")

osmium_version = subprocess.run([osmium_binary, "--version"], check=True, capture_output=True, text=True)
print(osmium_version.stdout.splitlines()[0])
print(f"pyogrio {pyogrio.__version__}, GDAL {pyogrio.__gdal_version_string__}")

In [ ]:
def tmp_path_for(path):
    # Outputs are written under a temporary name and renamed when complete, so an
    # aborted run never leaves a truncated file that a later run would skip over.
    return path.with_name(path.name + ".tmp")


def download_geofabrik_pbf(file_path, base_url="https://download.geofabrik.de/europe/"):
    if file_path.exists():
        print(f"File already exists: {file_path}, skipping download.")
        return

    file_url = base_url + file_path.name
    print(f"Downloading: {file_url}")
    tmp_path = tmp_path_for(file_path)
    with requests.get(file_url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(tmp_path, "wb") as file_handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                file_handle.write(chunk)
    tmp_path.replace(file_path)
    print(f"Downloaded: {file_path}")


def run_osmium(arguments):
    command = [osmium_binary, *arguments]
    print("Running:", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"osmium failed:\n{result.stderr}")
    return result.stdout


def osmium_tags_filter(input_pbf, output_pbf, expressions):
    tmp_path = tmp_path_for(output_pbf)
    run_osmium(["tags-filter", str(input_pbf), *expressions, "-o", str(tmp_path), "-f", "pbf", "--overwrite"])
    tmp_path.replace(output_pbf)


def write_osmconf(config_path, attributes):
    # Column definition for the GDAL OSM driver; tag `a:b` becomes column `a_b`.
    # Generated here because *.ini files are not tracked in git.
    config_path.write_text(
        "[lines]\n"
        "osm_id=yes\n"
        f"attributes={','.join(attributes)}\n"
        "other_tags=no\n"
    )


def pbf_to_parquet(input_pbf, output_parquet, osmconf_path, layer="lines"):
    gdf = pyogrio.read_dataframe(
        input_pbf,
        layer=layer,
        use_arrow=True,
        CONFIG_FILE=str(osmconf_path),
        MAX_TMPFILE_SIZE=4096,  # MB; keeps the node index in memory instead of a temp file
    )
    tmp_path = tmp_path_for(output_parquet)
    gdf.to_parquet(tmp_path)
    tmp_path.replace(output_parquet)
    return len(gdf)


def extract_ways(name, expressions, attributes):
    output_parquet = folder_processed / f"processed_{name}_germany_{set_date}.parquet"
    if output_parquet.exists() and not overwrite:
        print(f"Processed file already exists: {output_parquet}, skipping processing.")
        return output_parquet

    filtered_pbf = folder_processed / f"processed_{name}_germany_{set_date}.pbf"
    osmconf_path = folder_processed / f"osmconf_{name}.ini"

    download_geofabrik_pbf(source_pbf)
    osmium_tags_filter(source_pbf, filtered_pbf, expressions)
    write_osmconf(osmconf_path, attributes)
    n_rows = pbf_to_parquet(filtered_pbf, output_parquet, osmconf_path)

    # Every way osmium kept must end up as one row; a mismatch means the conversion lost data.
    n_ways = int(run_osmium(["fileinfo", "-e", "-g", "data.count.ways", str(filtered_pbf)]))
    if n_rows != n_ways:
        raise RuntimeError(f"{name}: {n_ways:,} ways in {filtered_pbf}, but {n_rows:,} rows in {output_parquet}")

    if not keep_filtered_pbf:
        filtered_pbf.unlink()

    print(f"{name}: {n_rows:,} ways -> {output_parquet}")
    return output_parquet

### Cycleways

Ein Filter für beide Kampagnen. Vor dem 20.09.2026 hatte jede ihren eigenen; die
Marking-Fassung kannte `sidewalk:bicycle` nicht und schrieb `w/bicycle=yes` und
`w/bicycle=designated` getrennt — bei osmium ist das Komma ein ODER, also derselbe
Ausdruck.

In [ ]:
cycleway_filter = [
    # Straßen mit potenzieller Radnutzung
    # "w/highway=cycleway,path,footway,residential,unclassified,living_street,road,pedestrian",
    "w/highway=cycleway,path,footway",

    # Wege, die explizit für Fahrräder ausgewiesen sind
    "w/bicycle=yes,designated",

    # Zusätzliche Radinfrastruktur-Tags
    "w/cycleway",
    "w/cycleway:left",
    "w/cycleway:right",
    "w/cycleway:both",
    "w/cycleway:lane",
    "w/cycleway:track",
    "w/cycleway:opposite",
    "w/cycleway:opposite_lane",
    "w/cycleway:opposite_track",
    "w/cycleway:shared_lane",
    # "w/cycleway:protected",

    # weiteres
    "w/sidewalk:bicycle",
    "w/sidewalk:left:bicycle",
    "w/sidewalk:right:bicycle",
    "w/sidewalk:both:bicycle",
    "w/bicycle:forward",
    "w/bicycle:backward",
]

cycleway_attributes = [
    "highway", "bicycle", "bicycle:forward", "bicycle:backward",
    "cycleway", "cycleway:left", "cycleway:right", "cycleway:both",
    "cycleway:lane", "cycleway:track", "cycleway:opposite", "cycleway:shared_lane",
    "sidewalk:bicycle", "sidewalk:right:bicycle", "sidewalk:left:bicycle", "sidewalk:both:bicycle",
    "maxspeed", "maxspeed:conditional", "maxspeed:backward", "maxspeed:forward", "maxspeed:type",
    "name", "ref", "surface", "width",
]

print(f"{len(cycleway_filter)} Filterausdrücke, {len(cycleway_attributes)} Attribute")
cycleways_parquet = extract_ways("cycleways", cycleway_filter, cycleway_attributes)

In [ ]:
parquet_metadata = pq.read_metadata(cycleways_parquet)
print(f"{cycleways_parquet}: {parquet_metadata.num_rows:,} rows")
print(parquet_metadata.schema.names)

### Motorways (optional)

Braucht selten eine Aktualisierung — die Autobahnen ändern sich kaum.

> **Achtung beim Erneuern.** Gelesen wird `utils/processed_motorways_germany_251215.parquet`,
> also eine Ebene höher als das, was dieses Notebook schreibt. Diese Datei ist in git
> getrackt und wird vom **wöchentlichen Server-Lauf** (`xb_`) gelesen. Ein neuer Stand
> muss also bewusst dorthin verschoben und der Dateiname in den lesenden Notebooks
> angepasst werden — nicht einfach hier erzeugen und hoffen.

In [ ]:
update_motorways = False

motorway_attributes = [
    "highway",
    "maxspeed", "maxspeed:conditional", "maxspeed:backward", "maxspeed:forward", "maxspeed:type",
    "name", "ref", "surface", "width",
]

if update_motorways:
    extract_ways("motorways", ["w/highway=motorway"], motorway_attributes)